# CommandLLM: Google Colab Training & Fine-Tuning Pipeline (Target > 98% Accuracy)
This notebook fine-tunes **CoreCommandLLM** (124M Custom Decoder Transformer) on domain-specific Linux & PowerShell sysadmin commands.

- **Base Architecture**: 12-layer decoder Transformer with weight tying (`wte` ↔ `lm_head`) and 50,263 vocabulary
- **Causal Alignment**: Standard $t \to t+1$ logit-target shifting with selective loss masking (`ignore_index=-100`)
- **Objective**: Achieve token & sequence validation accuracy > 98.0%
- **Hardware**: NVIDIA GPU with Automatic Mixed Precision (AMP)

In [1]:
# Cell 1: Environment & GPU Verification
!nvidia-smi

import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name    : {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    print(f'BF16 Supported : {torch.cuda.is_bf16_supported()}')

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


PyTorch Version: 2.12.0+cpu
CUDA Available : False


### Cell 2: Project Setup & Workspace Directory
Upload your project zip (`commandllm.zip`) to Colab or clone your repository.

In [ ]:
import os
import sys

workspace_dir = '/content/commandllm'
os.makedirs(workspace_dir, exist_ok=True)

# Handles either zip filename
zip_path = '/content/commandllm_colab.zip' if os.path.exists('/content/commandllm_colab.zip') else '/content/commandllm.zip'
!unzip -q -o {zip_path} -d {workspace_dir}

%cd /content/commandllm

if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

os.makedirs('checkpoints', exist_ok=True)
print(f"Working Directory: {os.getcwd()}")
!ls -la

Current working directory: c:\Users\SAMSUNG\OneDrive\Desktop\My-Projects\commandllm


'ls' is not recognized as an internal or external command,
operable program or batch file.


### Cell 3: Install Required Dependencies
Installs dependencies from `colab_requirements.txt`.

In [3]:
# Cell 3: Install dependencies
!pip install -r colab_requirements.txt

     ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
     --- ------------------------------------ 1.6/15.8 MB 12.0 MB/s eta 0:00:02
     -------------- ------------------------- 5.8/15.8 MB 16.7 MB/s eta 0:00:01
     ----------------------- ---------------- 9.2/15.8 MB 16.9 MB/s eta 0:00:01
     ------------------------------- ------- 12.6/15.8 MB 16.8 MB/s eta 0:00:01
     ---------------------------------------- 15.8/15.8 MB 16.8 MB/s  0:00:01
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): still running...
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached datasets-5.0.1-py3-none

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-grpc 1.42.1 requires opentelemetry-exporter-otlp-proto-common==1.42.1, but you have opentelemetry-exporter-otlp-proto-common 1.44.0 which is incompatible.
opentelemetry-exporter-otlp-proto-grpc 1.42.1 requires opentelemetry-proto==1.42.1, but you have opentelemetry-proto 1.44.0 which is incompatible.
opentelemetry-exporter-otlp-proto-grpc 1.42.1 requires opentelemetry-sdk~=1.42.1, but you have opentelemetry-sdk 1.44.0 which is incompatible.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Cell 3.5: Port Base GPT-2 Weights to CoreCommandLLM Checkpoint
Downloads pre-trained GPT-2 (124M) weights from Hugging Face and initializes `checkpoints/custom_base_checkpoint.pt` directly on the Colab instance in ~30 seconds (no 500MB upload needed).

In [4]:
# Cell 3.5: Build custom base checkpoint from Hugging Face GPT-2
!python scripts/port_weights.py

 CommandLLM Weight Porting Pipeline (HF GPT-2 -> CoreCommandLLM)
Instantiating custom CoreCommandLLM with config:
  CommandLMConfig(vocab_size=50263, block_size=256, n_layer=12, n_head=12, n_embd=768, dropout=0.1, bias=True)

Fetching pre-trained 'gpt2' weights from Hugging Face...
HF weights loaded successfully (149 keys in state_dict).

[wte] Ported 50,257 base tokens + initialized 6 special tokens (shape: torch.Size([50263, 768]))
[wpe] Sliced context window from 1024 to 256 tokens (shape: torch.Size([256, 768]))
[blocks] Ported and transposed weights across all 12 Transformer blocks.
[ln_f] Ported final LayerNorm parameters.
[lm_head] Tied lm_head.weight directly to transformer.wte.weight.

Loading mapped state dict into custom CoreCommandLLM (strict=True)...
Load result: missing_keys=[], unexpected_keys=[]

Serializing ported model checkpoint to: checkpoints\custom_base_checkpoint.pt
Checkpoint saved successfully! File size: 475.53 MB (498,624,313 bytes)

-------------------------


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2370.48it/s]


### Cell 4: Launch Fine-Tuning Pipeline (Target Accuracy > 98%)
Trains CoreCommandLLM with AdamW, Cosine LR scheduling with linear warmup, token-level accuracy tracking, and automatic checkpointing on best validation accuracy record.

In [5]:
# Cell 4: Execute fine-tuning pipeline with calibrated hyperparameters
!python -u scripts/train.py \
    --epochs 5 \
    --batch_size 16 \
    --learning_rate 5e-5 \
    --min_lr 5e-6 \
    --warmup_steps 250 \
    --weight_decay 0.01 \
    --grad_clip 1.0 \
    --checkpoint_path checkpoints/custom_base_checkpoint.pt \
    --save_path checkpoints/terminal_model_final.pt

^C


### Cell 5: Final Validation Benchmark (20 Unseen Prompts Evaluation)
Evaluates model generation against 20 unseen validation prompts, compares generated commands against ground truth, and outputs formal benchmark accuracy.

In [ ]:
# Cell 5: Benchmark evaluation on 20 unseen validation prompts
import os
import json
import torch
from core.config import CommandLMConfig
from core.model import CoreCommandLLM
from core.tokenizer import CommandTokenizer
from scripts.train import check_argument_preservation, normalize_command

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running Benchmark on Device: {device}')

# Load model and tokenizer
config = CommandLMConfig()
model = CoreCommandLLM(config)
ckpt_path = 'checkpoints/terminal_model_final.pt'
state_dict = torch.load(ckpt_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.to(device)
model.eval()
tokenizer = CommandTokenizer()

# Read 20 unseen validation samples
val_samples = []
val_file = 'data/val.jsonl'
with open(val_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            val_samples.append(json.loads(line))
        if len(val_samples) >= 20:
            break

print('=' * 95)
print(f' {"OS":<10} | {"PROMPT":<30} | {"MATCH":<5} | {"ARGS":<5} | {"GENERATED COMMAND"}')
print('=' * 95)

total_eval = len(val_samples)
exact_matches = 0
norm_matches = 0
arg_preserved = 0
token_correct_sum = 0
token_total_sum = 0

for idx, sample in enumerate(val_samples, 1):
    os_type = sample['os']
    prompt = sample['prompt']
    ground_truth = sample['cmd'].strip()

    prefix = tokenizer.format_prompt(os_type, prompt)
    input_ids = torch.tensor([tokenizer.encode(prefix)], dtype=torch.long, device=device)

    with torch.no_grad():
        out = model.generate(
            input_ids,
            max_new_tokens=48,
            temperature=0.0,
            top_k=None,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.0,
        )

    gen_tokens = out[0, len(input_ids[0]):].tolist()
    raw_decoded = tokenizer.decode(gen_tokens)
    pred_cmd = raw_decoded.split('<|end|>')[0].strip()

    is_match = (pred_cmd == ground_truth or pred_cmd.lower() == ground_truth.lower())
    if is_match:
        exact_matches += 1

    if normalize_command(pred_cmd).lower() == normalize_command(ground_truth).lower():
        norm_matches += 1

    ok_args, missing_args = check_argument_preservation(prompt, pred_cmd)
    if ok_args:
        arg_preserved += 1

    # Token-level overlap
    gt_tokens = tokenizer.encode(ground_truth)
    pred_toks = tokenizer.encode(pred_cmd)
    min_len = min(len(gt_tokens), len(pred_toks))
    t_correct = sum(1 for i in range(min_len) if gt_tokens[i] == pred_toks[i])
    token_correct_sum += t_correct
    token_total_sum += max(len(gt_tokens), len(pred_toks), 1)

    status_sym = '✓' if is_match else '~'
    arg_sym = '✓' if ok_args else '✗'
    prompt_trunc = (prompt[:28] + '...') if len(prompt) > 30 else prompt
    print(f' {os_type:<10} | {prompt_trunc:<30} | {status_sym:^5} | {arg_sym:^5} | {pred_cmd}')
    if not is_match:
        print(f' {"":<10} | {"  -> Expected:":<30} | {"":^5} | {"":^5} | {ground_truth}')

accuracy = (token_correct_sum / max(1, token_total_sum)) * 100.0
seq_acc = (exact_matches / total_eval) * 100.0
norm_acc = (norm_matches / total_eval) * 100.0
arg_acc = (arg_preserved / total_eval) * 100.0
print('=' * 95)
print(f' Exact Sequence Matches  : {exact_matches}/{total_eval} ({seq_acc:.1f}%)')
print(f' Normalized Exact Matches: {norm_matches}/{total_eval} ({norm_acc:.1f}%)')
print(f' Argument Preservation   : {arg_preserved}/{total_eval} ({arg_acc:.1f}%)')
print(f' Token-Level Accuracy    : {accuracy:.2f}%')
print('=' * 95)

### Cell 6: Save / Download Trained Model Checkpoint
Download `checkpoints/terminal_model_final.pt` directly or copy to Google Drive.

In [ ]:
# Cell 6: Download or save checkpoint to Google Drive
import os
from google.colab import files

ckpt_path = 'checkpoints/terminal_model_final.pt'
if os.path.exists(ckpt_path):
    size_mb = os.path.getsize(ckpt_path) / (1024 * 1024)
    print(f'Checkpoint verified: {ckpt_path} ({size_mb:.2f} MB)')
    
    # Option A: Download directly to your local computer
    print('Triggering browser download...')
    files.download(ckpt_path)
    
    # Option B: Uncomment below to save to Google Drive
    # from google.colab import drive
    # drive.mount('/content/drive')
    # !cp checkpoints/terminal_model_final.pt /content/drive/MyDrive/
else:
    print(f'Error: {ckpt_path} does not exist. Check training output.')